# Phase 1B — Combined Baselines: MOSAIKS + ResNet-18 + DINOv2 (Aggregate-Label)
**Runtime:** Google Colab — **GPU (T4 or A100)**

**Purpose:** Unified notebook combining NB11 (MOSAIKS + Ridge), NB12 (ResNet-18 + Ridge),
and NB13 (DINOv2 + Ridge) into a single run. Data is loaded **once** and shared across
all three feature extractors, eliminating redundant I/O.

Three frozen feature extractors are compared:

| Model | Dim | Notes |
|---|---|---|
| MOSAIKS (random conv) | 4096 | Rolf et al. 2021 random features |
| ResNet-18 (ImageNet) | 512 | 6-band adapted frozen backbone |
| DINOv2 ViT-Base | 768 | RGB band selection, CLS token |

Each is paired with `RidgeCV` regression on two aggregate targets:

| Target | Formula | Interpretation |
|---|---|---|
| `coverage_fraction` | n_ookla_tiles / total_possible_zoom16_tiles | Spatial coverage |
| `log_mean_tests` | log(1 + total_tests / n_subtiles) | Test density per sub-tile |

Predictions are binarised with a val-optimal threshold and evaluated against binary
ground-truth labels on the same Northeast India geographic hold-out.

**Diagnostic plots per model × target:**
1. Regularisation curve (train vs val MSE across alpha)
2. Predicted vs actual + Residual histogram
3. Residuals vs true value
4. Spatial prediction map
5. Confusion matrix + PR curve

**Inputs:**
- `data/raw/patches_2019/` — 6,970 GeoTIFF patches (224×224×6 HLS bands)
- `data/processed/sampled_patches.csv` — patch metadata + binary labels
- `data/raw/ookla_india_2019_Q1.csv` — raw Ookla sub-tile data

## Step 0: Environment Setup

In [ ]:
import os

REPO = 'satellite-internet-prediction-ML'
if not os.getcwd().endswith(REPO):
    if os.path.exists(REPO):
        %cd {REPO}
    else:
        !git clone https://github.com/tatyana21111/{REPO}.git
        %cd {REPO}
!git pull

In [ ]:
!pip install -q rasterio scikit-learn geopandas matplotlib seaborn scipy pyarrow joblib

In [ ]:
# Cell 0.3: patch check
from pathlib import Path

PATCH_DIR = 'data/raw/patches_2019'
patch_count = len(list(Path(PATCH_DIR).glob('*.tif')))
assert patch_count >= 6900, f'Only {patch_count} patches found — run NB01 first to download them.'
print(f'Patches available: {patch_count:,}')

In [ ]:
# sync patches from google drive
from google.colab import drive
import shutil

drive.mount('/content/drive', force_remount=False)

PATCH_DIR = 'data/raw/patches_2019'
Path(PATCH_DIR).mkdir(parents=True, exist_ok=True)

local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
if local_count < 6900:
    print('Syncing patches from Google Drive ...')
    drive_dir = '/content/drive/MyDrive/patches_2019'
    for f in Path(drive_dir).glob('*.tif'):
        dst = Path(PATCH_DIR) / f.name
        if not dst.exists():
            shutil.copy(f, dst)
    local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
print(f'Patches available: {local_count:,}')

## Step 1: Imports & Constants

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import matplotlib.pyplot as plt
import warnings
import logging
import json
import joblib
from pathlib import Path
from shapely.geometry import box
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score, precision_recall_curve,
    roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score, confusion_matrix,
    classification_report, mean_absolute_error, mean_squared_error,
)
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr, binned_statistic
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
logging.getLogger('rasterio').setLevel(logging.ERROR)

# ── Deterministic seeding ────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

HLS_MEANS = [0.14245495, 0.13921481, 0.12434631, 0.31420089, 0.20743526, 0.12046503]
HLS_STDS  = [0.04036231, 0.04186983, 0.05267646, 0.08222210, 0.06834774, 0.05294205]
SCALE     = 0.0001

PATCH_DIR   = 'data/raw/patches_2019'
RESULTS_DIR = Path('outputs/results')
FIGURES_DIR = Path('outputs/figures')
MODELS_DIR  = Path('outputs/models')
for p in [RESULTS_DIR, FIGURES_DIR, MODELS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PATCH_SIZE_M   = 6720.0
PATCH_SIZE_DEG = PATCH_SIZE_M / 111_320.0
EARTH_CIRC_M   = 40_075_016.686
ZOOM16_SIDE_EQ = EARTH_CIRC_M / (2 ** 16)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## Step 2: Load Data & Train / Val / Test Split
Pre-computed by NB01 (`patches_with_splits.csv`): binary labels,
aggregate targets (`coverage_fraction`, `log_mean_tests`),
stratification features, and geographic split.

In [ ]:
sampled = pd.read_csv('data/processed/patches_with_splits.csv')

# Filter to patches with available TIF files
available = set(f.stem for f in Path(PATCH_DIR).glob('*.tif'))
sampled = sampled[sampled['patch_id'].isin(available)].reset_index(drop=True)

train_df = sampled[sampled['split'] == 'train'].reset_index(drop=True)
val_df   = sampled[sampled['split'] == 'val'].reset_index(drop=True)
test_df  = sampled[sampled['split'] == 'test'].reset_index(drop=True)

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test (NE): {len(test_df):,}')
print(f'Test binary positive rate   : {test_df["has_coverage"].mean():.2%}')
print(f'Train coverage_fraction mean: {train_df["coverage_fraction"].mean():.4f}')
print(f'Test  coverage_fraction mean: {test_df["coverage_fraction"].mean():.4f}')

## Step 5: Feature Extraction — All Three Models

### 5A: MOSAIKS Random Convolutional Features (4096-d)

In [ ]:
class PatchDataset(Dataset):
    def __init__(self, df, patch_dir, n_bands=6):
        self.df        = df.reset_index(drop=True)
        self.patch_dir = patch_dir
        self.n_bands   = n_bands

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = f"{self.patch_dir}/{row['patch_id']}.tif"
        with rasterio.open(path) as src:
            img = src.read(list(range(1, self.n_bands + 1))).astype(np.float32)
        img *= SCALE
        for b in range(self.n_bands):
            img[b] = (img[b] - HLS_MEANS[b]) / HLS_STDS[b]
        img = np.nan_to_num(img, nan=0.0)
        return torch.from_numpy(img), int(row['has_coverage'])

In [ ]:
class MOSAIKSFeaturizer:
    """Random Convolutional Features (Rolf et al. 2021)."""
    def __init__(self, n_features=4096, patch_size=3, n_channels=6, seed=42, filter_chunk=256):
        rng     = np.random.RandomState(seed)
        filters = rng.randn(n_features, n_channels, patch_size, patch_size).astype(np.float32)
        self.filters      = torch.from_numpy(filters).to(DEVICE)
        self.bias         = 1.0
        self.filter_chunk = filter_chunk

    def featurize_batch(self, img_batch):
        img_batch = img_batch.to(DEVICE)
        all_out   = []
        with torch.no_grad():
            for i in range(0, self.filters.shape[0], self.filter_chunk):
                chunk = self.filters[i : i + self.filter_chunk]
                out   = F.conv2d(img_batch, chunk, padding=1)
                out   = torch.relu(out + self.bias)
                out   = out.mean(dim=[2, 3])
                all_out.append(out.cpu())
        return torch.cat(all_out, dim=1).numpy()


def extract_mosaiks(df, patch_dir, mosaiks_model, batch_size=64, desc=''):
    dataset = PatchDataset(df, patch_dir)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    feats, labels = [], []
    for imgs, lbls in tqdm(loader, desc=desc):
        feats.append(mosaiks_model.featurize_batch(imgs))
        labels.extend(lbls.numpy())
    return np.concatenate(feats, axis=0), np.array(labels)


mosaiks = MOSAIKSFeaturizer(n_features=4096, filter_chunk=256)
print('MOSAIKS featurizer ready — extracting ...')

M_train, y_train_bin = extract_mosaiks(train_df, PATCH_DIR, mosaiks, desc='MOSAIKS Train')
M_val,   y_val_bin   = extract_mosaiks(val_df,   PATCH_DIR, mosaiks, desc='MOSAIKS Val')
M_test,  y_test_bin  = extract_mosaiks(test_df,  PATCH_DIR, mosaiks, desc='MOSAIKS Test')
print(f'MOSAIKS feature shape: {M_train.shape}')

### 5B: ResNet-18 Frozen Features (512-d)

In [ ]:
class ResNet18Featurizer(nn.Module):
    """Frozen ResNet-18 with first conv adapted for 6-band HLS input."""
    def __init__(self, n_input_bands=6):
        super().__init__()
        resnet   = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        old_conv = resnet.conv1
        new_conv = nn.Conv2d(n_input_bands, 64, kernel_size=7,
                             stride=2, padding=3, bias=False)
        with torch.no_grad():
            new_conv.weight[:, :3] = old_conv.weight
            new_conv.weight[:, 3:] = (
                old_conv.weight.mean(dim=1, keepdim=True)
                .expand(-1, n_input_bands - 3, -1, -1)
            )
        resnet.conv1  = new_conv
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self.feat_dim = 512
        for param in self.backbone.parameters():
            param.requires_grad = False

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        return features.squeeze(-1).squeeze(-1)


def extract_nn_features(df, patch_dir, featurizer, batch_size=64, desc=''):
    dataset = PatchDataset(df, patch_dir)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    feats, labels = [], []
    for imgs, lbls in tqdm(loader, desc=desc):
        feats.append(featurizer(imgs.to(DEVICE)).cpu().numpy())
        labels.extend(lbls.numpy())
    return np.concatenate(feats, axis=0), np.array(labels)


resnet_feat = ResNet18Featurizer(n_input_bands=6).to(DEVICE).eval()
print(f'ResNet-18 ready — output dim: {resnet_feat.feat_dim}')

R_train, _ = extract_nn_features(train_df, PATCH_DIR, resnet_feat, desc='ResNet Train')
R_val,   _ = extract_nn_features(val_df,   PATCH_DIR, resnet_feat, desc='ResNet Val')
R_test,  _ = extract_nn_features(test_df,  PATCH_DIR, resnet_feat, desc='ResNet Test')
print(f'ResNet-18 feature shape: {R_train.shape}')

del resnet_feat
torch.cuda.empty_cache()

### 5C: DINOv2 ViT-Base Frozen Features (768-d)

In [ ]:
class DINOv2Featurizer(nn.Module):
    """Frozen DINOv2 ViT-Base with RGB band selection from 6-band HLS."""
    def __init__(self):
        super().__init__()
        self.dino     = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14',
                                       pretrained=True, verbose=False)
        self.rgb_idx  = [2, 1, 0]
        self.feat_dim = 768
        for param in self.dino.parameters():
            param.requires_grad = False

    def forward(self, x):
        x_rgb = x[:, self.rgb_idx, :, :]
        with torch.no_grad():
            return self.dino(x_rgb)


print('Loading DINOv2 ViT-Base ...')
dino_feat = DINOv2Featurizer().to(DEVICE).eval()
print(f'DINOv2 ready — output dim: {dino_feat.feat_dim}')

D_train, _ = extract_nn_features(train_df, PATCH_DIR, dino_feat, batch_size=32, desc='DINOv2 Train')
D_val,   _ = extract_nn_features(val_df,   PATCH_DIR, dino_feat, batch_size=32, desc='DINOv2 Val')
D_test,  _ = extract_nn_features(test_df,  PATCH_DIR, dino_feat, batch_size=32, desc='DINOv2 Test')
print(f'DINOv2 feature shape: {D_train.shape}')

del dino_feat
torch.cuda.empty_cache()

### 5D: Scale Features & Build Feature Registry

In [ ]:
# MOSAIKS: no scaling (same as NB11)
M_trainval = np.concatenate([M_train, M_val], axis=0)

# ResNet-18: StandardScaler (same as NB12)
R_trainval = np.concatenate([R_train, R_val], axis=0)
scaler_r   = StandardScaler()
R_trainval_sc = scaler_r.fit_transform(R_trainval)
R_train_sc    = scaler_r.transform(R_train)
R_val_sc      = scaler_r.transform(R_val)
R_test_sc     = scaler_r.transform(R_test)

# DINOv2: StandardScaler (same as NB13)
D_trainval = np.concatenate([D_train, D_val], axis=0)
scaler_d   = StandardScaler()
D_trainval_sc = scaler_d.fit_transform(D_trainval)
D_train_sc    = scaler_d.transform(D_train)
D_val_sc      = scaler_d.transform(D_val)
D_test_sc     = scaler_d.transform(D_test)

FEATURE_REGISTRY = {
    'MOSAIKS': {
        'X_train': M_train, 'X_val': M_val, 'X_test': M_test,
        'X_trainval': M_trainval,
        'dim': M_train.shape[1],
    },
    'ResNet-18': {
        'X_train': R_train_sc, 'X_val': R_val_sc, 'X_test': R_test_sc,
        'X_trainval': R_trainval_sc,
        'dim': R_train.shape[1],
    },
    'DINOv2': {
        'X_train': D_train_sc, 'X_val': D_val_sc, 'X_test': D_test_sc,
        'X_trainval': D_trainval_sc,
        'dim': D_train.shape[1],
    },
}

for name, fd in FEATURE_REGISTRY.items():
    print(f'{name:12s}: {fd["dim"]:5d}-d  |  train+val={fd["X_trainval"].shape[0]}  test={fd["X_test"].shape[0]}')

## Step 6: Evaluation Infrastructure

Shared function that trains a RidgeCV, generates all diagnostic plots, and returns metrics.

In [ ]:
ALPHAS = [0.01, 0.1, 1, 10, 100, 1000]
TARGETS = ['coverage_fraction', 'log_mean_tests']


def train_and_evaluate(model_name, feat_data, target_name,
                       train_df, val_df, test_df):
    """Full pipeline: Ridge fit, diagnostic plots, binary metrics."""
    safe_name = model_name.lower().replace('-', '').replace(' ', '_')
    tag = f'{safe_name}_agg_{target_name}'

    X_trainval = feat_data['X_trainval']
    X_train    = feat_data['X_train']
    X_val      = feat_data['X_val']
    X_test     = feat_data['X_test']

    y_trainval = np.concatenate([
        train_df[target_name].values, val_df[target_name].values
    ])
    y_train = train_df[target_name].values
    y_val   = val_df[target_name].values
    y_test  = test_df[target_name].values

    y_val_bin  = val_df['has_coverage'].values
    y_test_bin = test_df['has_coverage'].values

    print(f'\n{"="*60}')
    print(f'{model_name} — {target_name}')
    print(f'{"="*60}')

    # ── Figure 1: Regularisation curve (alpha sweep) ─────
    train_mse_by_alpha = []
    val_mse_by_alpha   = []
    for alpha in ALPHAS:
        ridge_tmp = Ridge(alpha=alpha)
        ridge_tmp.fit(X_train, y_train)
        train_mse_by_alpha.append(mean_squared_error(y_train, ridge_tmp.predict(X_train)))
        val_mse_by_alpha.append(mean_squared_error(y_val, ridge_tmp.predict(X_val)))

    best_alpha_idx = int(np.argmin(val_mse_by_alpha))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(ALPHAS, train_mse_by_alpha, 'o-', lw=1.5, color='#55A868', label='Train MSE')
    ax.plot(ALPHAS, val_mse_by_alpha,   'o-', lw=2,   color='#4C72B0', label='Val MSE')
    ax.axvline(ALPHAS[best_alpha_idx], linestyle='--', color='red', alpha=0.8,
               label=f'Best \u03b1={ALPHAS[best_alpha_idx]}  (Val MSE={val_mse_by_alpha[best_alpha_idx]:.4f})')
    ax.set_xscale('log')
    ax.set_xlabel('Ridge \u03b1 (regularisation strength)'); ax.set_ylabel('MSE')
    ax.set_title(f'Regularisation Curve — {model_name} — {target_name}')
    ax.legend(); plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{tag}_alpha_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Fit final RidgeCV on train+val ───────────────────
    ridge = RidgeCV(alphas=ALPHAS)
    ridge.fit(X_trainval, y_trainval)
    print(f'Best alpha (RidgeCV): {ridge.alpha_}')

    # ── Predictions ──────────────────────────────────────
    y_pred_test = ridge.predict(X_test)
    y_pred_val  = ridge.predict(X_val)
    residuals   = y_pred_test - y_test

    # ── Regression metrics ───────────────────────────────
    mae  = mean_absolute_error(y_test, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    rho, _ = spearmanr(y_test, y_pred_test)

    print(f'  MAE={mae:.4f}  RMSE={rmse:.4f}  Spearman \u03c1={rho:.4f}')

    # ── Figure 2: Predicted vs actual + Residual histogram
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f'Regression Quality — {model_name} — {target_name}',
                 fontsize=13, fontweight='bold')

    axes[0].scatter(y_test, y_pred_test, alpha=0.35, s=12, color='#4C72B0')
    mn = min(y_test.min(), y_pred_test.min())
    mx = max(y_test.max(), y_pred_test.max())
    axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='45\u00b0 reference')
    axes[0].set_xlabel(f'Actual {target_name}')
    axes[0].set_ylabel(f'Predicted {target_name}')
    axes[0].set_title(
        f'Predicted vs Actual\nMAE={mae:.3f}  RMSE={rmse:.3f}  Spearman \u03c1={rho:.3f}')
    axes[0].legend(fontsize=9)

    axes[1].hist(residuals, bins=40, color='#4C72B0', edgecolor='white', alpha=0.8)
    axes[1].axvline(0, color='red', linestyle='--', lw=1.5, label='Zero')
    axes[1].axvline(residuals.mean(), color='orange', linestyle='-', lw=1.5,
                    label=f'Mean = {residuals.mean():.3f}')
    axes[1].set_xlabel('Residual (predicted \u2212 true)')
    axes[1].set_ylabel('Count')
    axes[1].set_title(f'Residual Distribution  (std = {residuals.std():.3f})')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{tag}_regression.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Figure 3: Residuals vs true value ────────────────
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(y_test, residuals, alpha=0.35, s=12, color='#4C72B0')
    ax.axhline(0, color='red', linestyle='--', lw=1.5, label='Zero residual')
    try:
        bin_means, bin_edges, _ = binned_statistic(
            y_test, residuals, statistic='mean', bins=10)
        bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
        ax.plot(bin_centers, bin_means, 'o-', color='orange', lw=2,
                ms=6, label='Bin mean residual')
    except Exception:
        pass
    ax.set_xlabel(f'True {target_name}')
    ax.set_ylabel('Residual  (predicted \u2212 true)')
    ax.set_title(f'Residuals vs True Value — {model_name} — {target_name}')
    ax.legend(); plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{tag}_residuals.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Figure 4: Spatial prediction map ─────────────────
    lons = test_df['lon'].values
    lats = test_df['lat'].values
    res_abs_max = np.percentile(np.abs(residuals), 95)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'Spatial Prediction Map — {model_name} — {target_name}\nNE India test set',
                 fontsize=13, fontweight='bold')

    sc0 = axes[0].scatter(lons, lats, c=y_pred_test, cmap='RdYlGn', s=18, alpha=0.75)
    plt.colorbar(sc0, ax=axes[0], label=f'Predicted {target_name}')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
    axes[0].set_title('Predicted value')

    sc1 = axes[1].scatter(lons, lats, c=residuals, cmap='RdBu_r', s=18, alpha=0.75,
                          vmin=-res_abs_max, vmax=res_abs_max)
    plt.colorbar(sc1, ax=axes[1], label='Residual (predicted \u2212 true)')
    axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')
    axes[1].set_title('Residuals  (blue = under-predicted, red = over-predicted)')

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{tag}_spatial.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Val-optimal binarisation threshold ───────────────
    prec_v, rec_v, thr_v = precision_recall_curve(y_val_bin, y_pred_val)
    f1_v    = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-8)
    opt_thr = float(thr_v[np.argmax(f1_v)])
    print(f'  Val-optimal threshold: {opt_thr:.4f}')

    y_pred_bin = (y_pred_test >= opt_thr).astype(int)

    pr_auc   = average_precision_score(y_test_bin, y_pred_test)
    roc_auc  = roc_auc_score(y_test_bin, y_pred_test)
    f1_opt   = f1_score(y_test_bin, y_pred_bin)
    prec_opt = precision_score(y_test_bin, y_pred_bin, zero_division=0)
    rec_opt  = recall_score(y_test_bin, y_pred_bin)
    acc_opt  = accuracy_score(y_test_bin, y_pred_bin)

    print(f'  PR-AUC={pr_auc:.4f}  ROC-AUC={roc_auc:.4f}  F1={f1_opt:.4f}')
    print(f'  Precision={prec_opt:.4f}  Recall={rec_opt:.4f}  Accuracy={acc_opt:.4f}')
    print()
    print(classification_report(y_test_bin, y_pred_bin,
                                 target_names=['No Coverage', 'Has Coverage']))

    # ── Figure 5: Confusion matrix + PR curve ────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Binary Connectivity Task — {model_name} — {target_name}',
                 fontsize=13, fontweight='bold')

    cm = confusion_matrix(y_test_bin, y_pred_bin)
    im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
    fig.colorbar(im, ax=axes[0])
    axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(['No Cov.', 'Has Cov.'])
    axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(['No Cov.', 'Has Cov.'])
    for row in range(2):
        for col in range(2):
            axes[0].text(col, row, str(cm[row, col]), ha='center', va='center',
                         color='white' if cm[row, col] > cm.max() / 2 else 'black', fontsize=14)
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    axes[0].set_title(f'Confusion Matrix  (thr = {opt_thr:.3f})')

    prec_t, rec_t, _ = precision_recall_curve(y_test_bin, y_pred_test)
    axes[1].plot(rec_t, prec_t, lw=2, label=f'PR-AUC = {pr_auc:.3f}')
    axes[1].axhline(y_test_bin.mean(), linestyle='--', color='grey',
                    label=f'Baseline ({y_test_bin.mean():.3f})')
    axes[1].scatter([rec_opt], [prec_opt], marker='*', s=200, color='red', zorder=5,
                    label=f'Val-opt thr = {opt_thr:.3f}')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    axes[1].set_title('Precision-Recall Curve')
    axes[1].legend(loc='upper right'); axes[1].set_xlim([0, 1]); axes[1].set_ylim([0, 1])

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{tag}_binary.png', dpi=150, bbox_inches='tight')
    plt.show()

    metrics = {
        'model':         f'{model_name}_ridge_agg_{target_name}',
        'target':        target_name,
        'n_features':    feat_data['dim'],
        'mae':           mae,
        'rmse':          rmse,
        'spearman_rho':  rho,
        'pr_auc':        pr_auc,
        'roc_auc':       roc_auc,
        'f1_opt':        f1_opt,
        'opt_threshold': opt_thr,
        'precision_opt': prec_opt,
        'recall_opt':    rec_opt,
        'accuracy_opt':  acc_opt,
        'best_alpha':    float(ridge.alpha_),
        'n_train':       len(train_df) + len(val_df),
        'n_test':        len(test_df),
    }

    return ridge, metrics


print('Evaluation infrastructure ready.')

## Step 7: Train & Evaluate — All Models × Both Targets

In [ ]:
all_results = {}
all_models  = {}

for model_name, feat_data in FEATURE_REGISTRY.items():
    for target in TARGETS:
        ridge, metrics = train_and_evaluate(
            model_name=model_name,
            feat_data=feat_data,
            target_name=target,
            train_df=train_df,
            val_df=val_df,
            test_df=test_df,
        )
        key = (model_name, target)
        all_results[key] = metrics
        all_models[key]  = ridge

print('\n\nAll experiments complete.')

## Step 8: Cross-Model Comparison

In [ ]:
comp_rows = []
for key, m in all_results.items():
    model_name, target = key
    comp_rows.append({
        'Model':      model_name,
        'Target':     target,
        'Dims':       m['n_features'],
        'MAE':        m['mae'],
        'RMSE':       m['rmse'],
        'Spearman \u03c1': m['spearman_rho'],
        'PR-AUC':     m['pr_auc'],
        'ROC-AUC':    m['roc_auc'],
        'F1':         m['f1_opt'],
        'Precision':  m['precision_opt'],
        'Recall':     m['recall_opt'],
        'Accuracy':   m['accuracy_opt'],
    })

comparison = pd.DataFrame(comp_rows).sort_values(['Target', 'Model']).reset_index(drop=True)

print('=== Combined Baselines — NE India Test Set ===\n')
print(comparison.to_string(index=False))

comparison.to_csv(RESULTS_DIR / 'combined_baselines_agg_comparison.csv', index=False)
print('\nComparison table saved.')

In [ ]:
for target in TARGETS:
    subset = comparison[comparison['Target'] == target].reset_index(drop=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Combined Baselines — {target}\nNE India test set',
                 fontsize=13, fontweight='bold')

    labels = subset['Model'].tolist()
    x = np.arange(len(labels))
    w = 0.2

    axes[0].bar(x - w, subset['MAE'],  w, label='MAE',  color='#4C72B0')
    axes[0].bar(x,     subset['RMSE'], w, label='RMSE', color='#DD8452')
    axes[0].bar(x + w, subset['Spearman \u03c1'].clip(0), w, label='Spearman \u03c1', color='#55A868')
    axes[0].set_xticks(x); axes[0].set_xticklabels(labels, rotation=10, ha='right')
    axes[0].set_ylabel('Score'); axes[0].set_title('Regression Metrics')
    axes[0].legend()

    axes[1].bar(x - w, subset['PR-AUC'],  w, label='PR-AUC',  color='#4C72B0')
    axes[1].bar(x,     subset['ROC-AUC'], w, label='ROC-AUC', color='#DD8452')
    axes[1].bar(x + w, subset['F1'],      w, label='F1',      color='#55A868')
    axes[1].set_xticks(x); axes[1].set_xticklabels(labels, rotation=10, ha='right')
    axes[1].set_ylim(0, 1); axes[1].set_ylabel('Score')
    axes[1].set_title('Binary Connectivity Metrics')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'combined_baselines_{target}_comparison.png',
                dpi=150, bbox_inches='tight')
    plt.show()

# Combined overview
fig, ax = plt.subplots(figsize=(15, 5))
labels = comparison['Model'] + ' \u2014 ' + comparison['Target']
x = np.arange(len(labels))
w = 0.25
ax.bar(x - w, comparison['PR-AUC'],  w, label='PR-AUC',  color='#4C72B0')
ax.bar(x,     comparison['ROC-AUC'], w, label='ROC-AUC', color='#DD8452')
ax.bar(x + w, comparison['F1'],      w, label='F1',      color='#55A868')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=18, ha='right')
ax.set_ylim(0, 1); ax.set_ylabel('Score')
ax.set_title('Combined Baselines: MOSAIKS vs ResNet-18 vs DINOv2 \u2014 All Targets\nNE India test set')
ax.legend(); plt.tight_layout()
plt.savefig(FIGURES_DIR / 'combined_baselines_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9: Save Metrics & Push to GitHub

In [ ]:
for key, m in all_results.items():
    model_name, target = key
    safe_name = model_name.lower().replace('-', '').replace(' ', '_')
    pd.DataFrame([m]).to_csv(
        RESULTS_DIR / f'{safe_name}_agg_{target}_metrics.csv', index=False)

for key, ridge in all_models.items():
    model_name, target = key
    safe_name = model_name.lower().replace('-', '').replace(' ', '_')
    joblib.dump(ridge, MODELS_DIR / f'{safe_name}_agg_{target}.pkl')

joblib.dump(scaler_r, MODELS_DIR / 'resnet18_agg_scaler.pkl')
joblib.dump(scaler_d, MODELS_DIR / 'dinov2_agg_scaler.pkl')

print('All metrics and models saved.')

In [ ]:
import subprocess, os

token = "YOUR_TOKEN_HERE"
repo  = "tatyana21111/satellite-internet-prediction-ML"

subprocess.run(['git', 'config', '--global', 'user.email', 'tatyana211zy@outlook.com'], check=True)
subprocess.run(['git', 'config', '--global', 'user.name',  'tatyana21111'], check=True)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/{repo}.git'], check=True)

# Auto-discover all output files for this notebook
fig_patterns = ['mosaiks_agg_*', 'resnet18_agg_*', 'dinov2_agg_*', 'combined_baselines_*']
files = ['notebooks/03_aggregate_baselines.ipynb']
for pat in fig_patterns:
    files.extend([str(p) for p in FIGURES_DIR.glob(pat)])
    files.extend([str(p) for p in RESULTS_DIR.glob(pat)])
    files.extend([str(p) for p in MODELS_DIR.glob(pat)])

for f in files:
    if os.path.exists(f):
        subprocess.run(['git', 'add', '-f', f], check=True)
    else:
        print(f'Skipping (not yet generated): {f}')

diff = subprocess.run(['git', 'diff', '--cached', '--quiet'], capture_output=True)
if diff.returncode != 0:
    subprocess.run(['git', 'commit', '-m',
                    'NB03: Aggregate-label baselines (MOSAIKS + ResNet-18 + DINOv2)'], check=True)
else:
    print('Nothing staged.')

subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], check=True)
subprocess.run(['git', 'push'], check=True)
print('Push successful.')